In [14]:
import pandas as pd

excel_file = 'SummaryReport.xlsx'

# 1. Load the raw sheets
df_basic = pd.read_excel(excel_file, sheet_name='Basic Info').drop_duplicates(subset=['Stockname'])
df_ratios = pd.read_excel(excel_file, sheet_name='Top Ratios').drop_duplicates(subset=['Stockname'])
df_balance_raw = pd.read_excel(excel_file, sheet_name='Balance Sheet')

# Clean names
df_basic['Stockname'] = df_basic['Stockname'].astype(str).str.strip()
df_ratios['Stockname'] = df_ratios['Stockname'].astype(str).str.strip()
df_balance_raw['Stockname'] = df_balance_raw['Stockname'].astype(str).str.strip()
df_balance_raw['Balance Sheet'] = df_balance_raw['Balance Sheet'].astype(str).str.strip()

# 2. Extract Mar 2024 values for our core balance sheet components
# Pivot or filter to get Borrowings, Equity Capital, and Reserves side-by-side
mar_2024_data = df_balance_raw[['Stockname', 'Balance Sheet', 'Mar 2024']]

# Create individual dataframes for each metric from the rows
borrowings = mar_2024_data[mar_2024_data['Balance Sheet'] == 'Borrowings'].rename(columns={'Mar 2024': 'Borrowings_Mar2024'})
equity = mar_2024_data[mar_2024_data['Balance Sheet'] == 'Equity Capital'].rename(columns={'Mar 2024': 'Equity_Mar2024'})
reserves = mar_2024_data[mar_2024_data['Balance Sheet'] == 'Reserves'].rename(columns={'Mar 2024': 'Reserves_Mar2024'})

# Merge the balance sheet components into a single clean flat dataframe
df_balance_clean = pd.merge(equity[['Stockname', 'Equity_Mar2024']], borrowings[['Stockname', 'Borrowings_Mar2024']], on='Stockname', how='outer')
df_balance_clean = pd.merge(df_balance_clean, reserves[['Stockname', 'Reserves_Mar2024']], on='Stockname', how='outer')

# 3. Combine everything into our final flat master dataframe
df_master = pd.merge(df_basic, df_ratios, on='Stockname', how='inner')
df_master = pd.merge(df_master, df_balance_clean, on='Stockname', how='inner')

print(f"MATRIX FOOTPRINT ESTABLISHED")
print(f"Successfully processed {len(df_master)} unique companies with flat balance sheet metrics.")
print(df_master[['Stockname', 'Equity_Mar2024', 'Borrowings_Mar2024', 'Reserves_Mar2024']].head(3))

MATRIX FOOTPRINT ESTABLISHED
Successfully processed 454 unique companies with flat balance sheet metrics.
  Stockname  Equity_Mar2024  Borrowings_Mar2024  Reserves_Mar2024
0  RELIANCE          6766.0            458991.0          786715.0
1       TCS           362.0              8021.0           90127.0
2  HDFCBANK           760.0           3107503.0          455636.0


In [12]:
# Group by sector and calculate average PE, ROE, and total market cap
sector_analysis = df_master.groupby('Sector').agg(
    Avg_PE=('Stock PE', 'mean'),
    Avg_ROE=('ROE %', 'mean'),
    Total_Market_Cap=('Market Cap in Cores Rupees', 'sum')
).reset_index()

# Calculate our Valuation Premium (PE divided by ROE)
sector_analysis['PE_to_ROE_Ratio'] = sector_analysis['Avg_PE'] / sector_analysis['Avg_ROE']

# Sort descending to find the absolute worst overvalued sectors at the top
valuation_traps = sector_analysis.sort_values(by='PE_to_ROE_Ratio', ascending=False)

print("=== TOP 5 VALUATION TRAP SECTORS (Highest PE to ROE Premium) ===")
print(valuation_traps[['Sector', 'Avg_PE', 'Avg_ROE', 'PE_to_ROE_Ratio']].head(5))

=== TOP 5 VALUATION TRAP SECTORS (Highest PE to ROE Premium) ===
                             Sector   Avg_PE  Avg_ROE  PE_to_ROE_Ratio
38                            Paper  1226.00    1.290       950.387597
44         Quick Service Restaurant   836.50    8.960        93.359375
16  E-Commerce/App based Aggregator   352.15    8.586        41.014442
49                           Retail   100.80    3.625        27.806897
31                        Logistics   250.35    9.880        25.339069


In [15]:
# Create a fresh copy for calculations
df_debt_analysis = df_master.copy()

# Fill any missing values with 0 to prevent math errors
df_debt_analysis['Borrowings_Mar2024'] = df_debt_analysis['Borrowings_Mar2024'].fillna(0)
df_debt_analysis['Equity_Mar2024'] = df_debt_analysis['Equity_Mar2024'].fillna(0)
df_debt_analysis['Reserves_Mar2024'] = df_debt_analysis['Reserves_Mar2024'].fillna(0)

# Formula: Debt / (Equity + Reserves)
df_debt_analysis['DE_Ratio'] = df_debt_analysis['Borrowings_Mar2024'] / (df_debt_analysis['Equity_Mar2024'] + df_debt_analysis['Reserves_Mar2024'])

# Filter out financial sectors and isolate operational companies with D/E > 3.0
danger_zone = df_debt_analysis[
    (~df_debt_analysis['Sector'].isin(['Banks', 'Finance', 'Financial Services'])) & 
    (df_debt_analysis['DE_Ratio'] > 3.0)
]

print(f"=== NON-FINANCIAL DEBT DANGER ZONE ({len(danger_zone)} Companies Found) ===")
print(danger_zone[['Stockname', 'Sector', 'DE_Ratio']].sort_values(by='DE_Ratio', ascending=False).head(10))

=== NON-FINANCIAL DEBT DANGER ZONE (18 Companies Found) ===
      Stockname                           Sector   DE_Ratio
314  CHOLAHLDNG                             Null  13.084749
351     TVSHLTD                             Null   9.236620
115    CHOLAFIN                             Null   6.863421
65   ADANIGREEN  Power Generation & Distribution   6.595282
224    TATACOMM                 Telecomm-Service   6.306271
64   BAJAJFINSV                             Null   4.789368
243  SUNDARMFIN                             Null   4.724138
189    ASHOKLEY                       Automobile   4.531038
409        IIFL                             Null   4.433675
343        ISEC                             Null   4.304614


In [16]:
# Apply our strict fundamental and valuation filters
hidden_gems = df_debt_analysis[
    (df_debt_analysis['ROE %'] > 20) &
    (df_debt_analysis['ROCE %'] > 15) &
    (df_debt_analysis['Stock PE'] < 25) &
    (df_debt_analysis['DE_Ratio'] < 1.0)
]

# Drop companies with missing sector classifications to keep our list pristine
hidden_gems_clean = hidden_gems.dropna(subset=['Sector'])

print(f"=== THE HISTORIC HIDDEN GEMS ({len(hidden_gems_clean)} Sector-Verified Companies Found) ===")
print(hidden_gems_clean[['Stockname', 'Sector', 'ROE %', 'Stock PE', 'DE_Ratio']].sort_values(by='ROE %', ascending=False))

=== THE HISTORIC HIDDEN GEMS (26 Sector-Verified Companies Found) ===
      Stockname                               Sector  ROE %  Stock PE  \
436   JAIBALAJI                                Steel   87.3     15.70   
7          LICI                                 Null   63.4     10.80   
67   ADANIPOWER      Power Generation & Distribution   57.1     14.50   
72     HINDZINC                   Non Ferrous Metals   55.2     17.80   
60    COALINDIA            Mining & Mineral products   52.0      6.53   
353        AIIL                                 Null   44.0      6.46   
367    ANGELONE                                 Null   43.3     13.60   
95         BPCL                           Refineries   41.9      7.74   
90      HYUNDAI                           Automobile   39.6     23.40   
311        MRPL                           Refineries   31.9     21.80   
427  SONATSOFTW                        IT - Software   30.3     24.60   
264   NAM-INDIA                                 Null  

In [17]:
# Let's inspect the names of the columns inside the growth sheets
df_sales_growth = pd.read_excel('SummaryReport.xlsx', sheet_name='Compounded Sales Growth')
df_profit_growth = pd.read_excel('SummaryReport.xlsx', sheet_name='Compounded Profit Growth')

print("Sales Growth Columns:")
print(list(df_sales_growth.columns))

print("\nProfit Growth Columns:")
print(list(df_profit_growth.columns))

Sales Growth Columns:
['Compounded Sales Growth', 'Stockname', 'Compounded Sales Growth.1']

Profit Growth Columns:
['Compounded Profit Growth', 'Stockname', 'Compounded Profit Growth.1']


In [18]:
# Let's peek at the first 10 rows of the sales growth sheet to see how rows are labeled
print("Sales Growth Sheet Row Preview:")
print(df_sales_growth.head(10))

Sales Growth Sheet Row Preview:
  Compounded Sales Growth Stockname  Compounded Sales Growth.1
0                10 Years  RELIANCE                        8.0
1                 5 Years  RELIANCE                       10.0
2                 3 Years  RELIANCE                       24.0
3                     TTM  RELIANCE                        7.0
4                10 Years       TCS                       11.0
5                 5 Years       TCS                       10.0
6                 3 Years       TCS                       14.0
7                     TTM       TCS                        6.0
8                10 Years  HDFCBANK                       21.0
9                 5 Years  HDFCBANK                       22.0


In [19]:
# 1. Clean the labels to avoid hidden whitespace bugs
df_sales_growth['Compounded Sales Growth'] = df_sales_growth['Compounded Sales Growth'].astype(str).str.strip()
df_profit_growth['Compounded Profit Growth'] = df_profit_growth['Compounded Profit Growth'].astype(str).str.strip()
df_sales_growth['Stockname'] = df_sales_growth['Stockname'].astype(str).str.strip()
df_profit_growth['Stockname'] = df_profit_growth['Stockname'].astype(str).str.strip()

# 2. Extract only the '5 Years' row for each company
sales_5y = df_sales_growth[df_sales_growth['Compounded Sales Growth'] == '5 Years'][['Stockname', 'Compounded Sales Growth.1']]
sales_5y = sales_5y.rename(columns={'Compounded Sales Growth.1': 'Sales_CAGR_5Y'})

profit_5y = df_profit_growth[df_profit_growth['Compounded Profit Growth'] == '5 Years'][['Stockname', 'Compounded Profit Growth.1']]
profit_5y = profit_5y.rename(columns={'Compounded Profit Growth.1': 'Profit_CAGR_5Y'})

# 3. Merge the growth metrics into a temporary dataframe
df_growth = pd.merge(sales_5y, profit_5y, on='Stockname', how='inner')

# 4. Bring the growth columns into our core master data frame
df_master_growth = pd.merge(df_debt_analysis, df_growth, on='Stockname', how='inner')

# 5. Calculate the Growth Quality Ratio
# If a company has negative or 0 sales growth, we handle it safely to avoid division by zero
df_master_growth['Growth_Quality'] = df_master_growth['Profit_CAGR_5Y'] / df_master_growth['Sales_CAGR_5Y']

# 6. Filter for India's Top 50 companies by Market Cap with Sales Growth > 10%
top_50_growth = df_master_growth.sort_values(by='Market Cap in Cores Rupees', ascending=False).head(50)
high_growth_top_50 = top_50_growth[top_50_growth['Sales_CAGR_5Y'] > 10]

# Isolate the hollow growth players (where Profit CAGR grew slower than Sales CAGR)
hollow_growth = high_growth_top_50[high_growth_top_50['Growth_Quality'] < 1.0]

print(f"=== GROWTH QUALITY PARADOX IN TOP 50 MARKET GIANTS ===")
print(f"Found {len(hollow_growth)} mega-cap companies expanding revenue faster than earnings.")
print("\nNotable Top 50 Hollow Growth Stories (Sorted by worst ratio):")
print(hollow_growth[['Stockname', 'Sector', 'Sales_CAGR_5Y', 'Profit_CAGR_5Y', 'Growth_Quality']].sort_values(by='Growth_Quality'))

=== GROWTH QUALITY PARADOX IN TOP 50 MARKET GIANTS ===
Found 2 mega-cap companies expanding revenue faster than earnings.

Notable Top 50 Hollow Growth Stories (Sorted by worst ratio):
   Stockname         Sector  Sales_CAGR_5Y  Profit_CAGR_5Y  Growth_Quality
42   HCLTECH  IT - Software           13.0             9.0        0.692308
5       INFY  IT - Software           13.0            11.0        0.846154


In [20]:
# Let's see how columns are structured in the shareholding sheet
df_shareholding = pd.read_excel('SummaryReport.xlsx', sheet_name='Shareholding Pattern')

print("Shareholding Pattern Columns:")
print(list(df_shareholding.columns))

print("\nRow Label Preview:")
print(df_shareholding.head(5))

Shareholding Pattern Columns:
['Shareholding Pattern', 'Stockname', 'Mar 2022', 'Jun 2022', 'Sep 2022', 'Dec 2022', 'Mar 2023', 'Jun 2023', 'Sep 2023', 'Dec 2023', 'Mar 2024', 'Jun 2024', 'Sep 2024', 'Dec 2024', 'Feb 2025', 'Jan 2025']

Row Label Preview:
  Shareholding Pattern Stockname  Mar 2022  Jun 2022  Sep 2022  Dec 2022  \
0            Promoters  RELIANCE     50.66     50.62     50.56     50.49   
1                 FIIs  RELIANCE     24.23     23.90     23.58     23.48   
2                 DIIs  RELIANCE     14.23     14.67     14.91     15.26   
3           Government  RELIANCE      0.17      0.17      0.16      0.16   
4               Public  RELIANCE     10.71     10.64     10.78     10.59   

   Mar 2023  Jun 2023  Sep 2023  Dec 2023  Mar 2024  Jun 2024  Sep 2024  \
0     50.41     50.39     50.27     50.30     50.31     50.33     50.24   
1     22.49     22.55     22.60     22.13     22.06     21.75     21.30   
2     16.06     16.13     15.99     16.59     16.98     17.30 

In [22]:
# 1. Clean the text flags to avoid whitespace mismatches
df_shareholding['Shareholding Pattern'] = df_shareholding['Shareholding Pattern'].astype(str).str.strip()
df_shareholding['Stockname'] = df_shareholding['Stockname'].astype(str).str.strip()

# 2. Isolate ONLY the Promoter rows
promoters_df = df_shareholding[df_shareholding['Shareholding Pattern'] == 'Promoters'].copy()

# 3. Drop missing values specifically in our baseline start and end periods to ensure accuracy
promoters_df = promoters_df.dropna(subset=['Mar 2022', 'Dec 2024'])

# 4. Calculate the net percentage change over the 2.5 year timeline
promoters_df['Promoter_Change'] = promoters_df['Dec 2024'] - promoters_df['Mar 2022']

# 5. Segment into clear confidence brackets
promoter_buyers = promoters_df[promoters_df['Promoter_Change'] >= 2.0]
promoter_sellers = promoters_df[promoters_df['Promoter_Change'] <= -2.0]

# 6. Calculate the core indicator metric
seller_to_buyer_ratio = len(promoter_sellers) / len(promoter_buyers) if len(promoter_buyers) > 0 else 0

print("=== PROMOTER CONFIDENCE INSIDER SIGNAL ===")
print(f"Companies with substantial promoter BUYING (>= 2% increase): {len(promoter_buyers)}")
print(f"Companies with substantial promoter SELLING (>= 2% decrease): {len(promoter_sellers)}")
print(f" Insider Selling-to-Buying Ratio: {seller_to_buyer_ratio:.1f}x")

print("\nTop 5 Insider Buying Companies (Promoters accumulating stake):")
print(promoter_buyers[['Stockname', 'Mar 2022', 'Dec 2024', 'Promoter_Change']].sort_values(by='Promoter_Change', ascending=False).head(5))

print("\nTop 5 Insider Selling Companies (Promoters liquidating stake):")
print(promoter_sellers[['Stockname', 'Mar 2022', 'Dec 2024', 'Promoter_Change']].sort_values(by='Promoter_Change').head(5))

=== PROMOTER CONFIDENCE INSIDER SIGNAL ===
Companies with substantial promoter BUYING (>= 2% increase): 19
Companies with substantial promoter SELLING (>= 2% decrease): 100
 Insider Selling-to-Buying Ratio: 5.3x

Top 5 Insider Buying Companies (Promoters accumulating stake):
      Stockname  Mar 2022  Dec 2024  Promoter_Change
1180    ESCORTS     28.10     68.04            39.94
1705    POLYMED     44.66     62.44            17.78
1790   TEJASNET     37.17     54.01            16.84
2099  JAIBALAJI     50.98     64.84            13.86
1648     526851     61.90     70.48             8.58

Top 5 Insider Selling Companies (Promoters liquidating stake):
       Stockname  Mar 2022  Dec 2024  Promoter_Change
1062     COFORGE     40.09      0.00           -40.09
1203    SONACOMS     67.18     28.03           -39.15
993   IDFCFIRSTB     36.49      0.00           -36.49
2040  IDFCFIRSTB     36.49      0.00           -36.49
868    PATANJALI     98.90     69.48           -29.42


In [23]:
# 1. Export the Master dataset containing Ratios, Valuation, and Debt metrics
df_master_growth.to_csv('master_nse.csv', index=False)

# 2. Export the Promoter trends dataset for ownership insights
promoters_df.to_csv('promoter_trends.csv', index=False)

print("EXPORT SUCCESSFUL!")
print("1. 'master_nse.csv' is saved and ready for SQL / Power BI.")
print("2. 'promoter_trends.csv' is saved and ready for insider trend analysis.")

EXPORT SUCCESSFUL!
1. 'master_nse.csv' is saved and ready for SQL / Power BI.
2. 'promoter_trends.csv' is saved and ready for insider trend analysis.


In [25]:
import sqlite3
import pandas as pd

# =====================================================================
# 1. DATABASE INITIALIZATION & DATA LOADING
# =====================================================================
# Spin up a local relational database connection
conn = sqlite3.connect('nse_analytics.db')
cursor = conn.cursor()

# Load the pristine datasets generated in your Python phase
df_master_nse = pd.read_csv('master_nse.csv')
df_promoter_trends = pd.read_csv('promoter_trends.csv')

# Write them into standard relational SQL tables
df_master_nse.to_sql('master_nse', conn, if_exists='replace', index=False)
df_promoter_trends.to_sql('promoter_trends', conn, if_exists='replace', index=False)

print(" SQL ENVIRONMENT ESTABLISHED SUCCESSFULY ")
print("Tables 'master_nse' and 'promoter_trends' are online.\n")
print("-" * 80 + "\n")


# =====================================================================
# QUERY 1: SECTOR COMPOSITE RANKING (WINDOW FUNCTIONS)
# =====================================================================
query_sector_ranking = """
with sector_metrics as (
    select 
        Sector,
        count(*) as Company_Count,
        avg("ROE %") as Avg_ROE,
        avg("ROCE %") as Avg_ROCE,
        avg("DE_Ratio") as Avg_DE_Ratio
    from master_nse
    where Sector is not null and Sector != 'Null'
    group by Sector
    having count(*) >= 3
),
ranked_sectors as (
    select 
        Sector,
        Company_Count,
        Avg_ROE,
        Avg_ROCE,
        Avg_DE_Ratio,
        rank() over (order by Avg_ROE desc) as ROE_Rank,
        rank() over (order by Avg_ROCE desc) as ROCE_Rank,
        rank() over (order by Avg_DE_Ratio asc) as Safety_Rank
    from sector_metrics
)
select 
    Sector,
    Company_Count,
    round(Avg_ROE, 2) as [Avg ROE %],
    round(Avg_ROCE, 2) as [Avg ROCE %],
    round(Avg_DE_Ratio, 2) as [Avg D/E],
    (ROE_Rank + ROCE_Rank + Safety_Rank) as Composite_Score
from ranked_sectors
order by Composite_Score asc;
"""

print("=== SQL QUERY 1: SECTOR COMPOSITE RANKING ===")
df_sql_result1 = pd.read_sql_query(query_sector_ranking, conn)
print(df_sql_result1.head(8))
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 2: INTRA-SECTOR VALUATION TRAPS (DENSE_RANK PARTITION)
# =====================================================================
query_valuation_traps = """
with ranked_traps as (
    select 
        Stockname,
        Sector,
        "Stock PE" as Stock_PE,
        "ROE %" as ROE,
        DE_Ratio,
        ("Stock PE" / case when "ROE %" <= 0 then 0.01 else "ROE %" end) as PE_to_ROE_Premium,
        dense_rank() over (partition by Sector order by ("Stock PE" / case when "ROE %" <= 0 then 0.01 else "ROE %" end) desc) as Intra_Sector_Rank
    from master_nse
    where Sector is not null and Sector != 'Null' and "Stock PE" is not null
)
select 
    Sector,
    Stockname,
    round(Stock_PE, 2) as [Stock P/E],
    round(ROE, 2) as [ROE %],
    round(PE_to_ROE_Premium, 2) as [P/E-to-ROE Premium]
from ranked_traps
where Intra_Sector_Rank <= 3 and [P/E-to-ROE Premium] > 10
order by Sector, [P/E-to-ROE Premium] desc;
"""

print("=== SQL QUERY 2: INTRA-SECTOR VALUATION TRAPS (TOP 3 PER SECTOR) ===")
df_sql_result2 = pd.read_sql_query(query_valuation_traps, conn)
print(df_sql_result2.head(10))
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 3: DEBT DANGER + INSIDER SELLING CONVERGENCE (RELATIONAL JOIN)
# =====================================================================
query_debt_plus_selling = """
select 
    m.Stockname,
    m.Sector,
    round(m.DE_Ratio, 2) as [D/E Ratio],
    round(p.Promoter_Change, 2) as [Promoter % Change Over 2.5Y]
from master_nse m
inner join promoter_trends p on m.Stockname = p.Stockname
where m.DE_Ratio > 3.0 
  and m.Sector not in ('Banks', 'Finance', 'Financial Services')
  and p.Promoter_Change < 0
order by m.DE_Ratio desc;
"""

print("=== SQL QUERY 3: DEBT DANGER ZONE + INSIDER SELLING CONVERGENCE ===")
df_sql_result3 = pd.read_sql_query(query_debt_plus_selling, conn)
print(df_sql_result3)
print("\n" + "-" * 80 + "\n")


# =====================================================================
# QUERY 4: SAFE GROWTH CHAMPIONS (FILTRATION & SELECTION)
# =====================================================================
query_growth_champions = """
select 
    Stockname,
    Sector,
    round(Sales_CAGR_5Y, 2) as [5Y Sales CAGR %],
    round(Profit_CAGR_5Y, 2) as [5Y Profit CAGR %],
    round(Growth_Quality, 2) as [Growth Quality Score],
    round("Stock PE", 2) as [Stock P/E]
from master_nse
where Growth_Quality >= 1.0 
  and "Stock PE" < 25 
  and DE_Ratio < 1.0
  and Sector is not null and Sector != 'Null'
order by Growth_Quality desc
limit 10;
"""

print("=== SQL QUERY 4: SAFE GROWTH CHAMPIONS ===")
df_sql_result4 = pd.read_sql_query(query_growth_champions, conn)
print(df_sql_result4)

 SQL ENVIRONMENT ESTABLISHED SUCCESSFULY 
Tables 'master_nse' and 'promoter_trends' are online.

--------------------------------------------------------------------------------

=== SQL QUERY 1: SECTOR COMPOSITE RANKING ===
                          Sector  Company_Count  Avg ROE %  Avg ROCE %  \
0             Financial Services              5      29.84       38.06   
1                           FMCG             14      36.11       50.06   
2                  IT - Software             55      24.46       30.14   
3                       Bearings              3      19.57       26.10   
4  Readymade Garments/ Apparells              4      24.32       27.65   
5                 Paints/Varnish              4      21.30       26.20   
6                Pharmaceuticals              9      21.82       27.40   
7                         Cables              4      19.60       24.73   

   Avg D/E  Composite_Score  
0     0.03               12  
1     0.30               20  
2     0.16        